# ⚽ HaxBall AI: Treinamento por Imitação Humana (Behavioral Cloning) no Google Colab

Este notebook executa o pipeline completo de Imitation Learning com Scraping Automático do Discord:
1. **Camada 1 (Scraping Automático):** Conecta à API do Discord e baixa os replays `.hbr2` oficiais.
2. **Camada 2 (Extração 61D):** Decodifica as partidas e gera observações egocêntricas completas de 61 dimensões com espelhamento Y.
3. **Camada 3 (Treino em GPU):** Treina a rede neural `HaxBallExpertPolicy` com AdamW + Cosine Annealing.
4. **Camada 4 (Export):** Download automático do arquivo `.pt` para jogar contra na GUI local!

In [ ]:
# 1. Hardware e GPU
import os, sys, time, torch
print(f'PyTorch Version: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo de Treinamento: {device}')
if torch.cuda.is_available():
    print(f'GPU Ativa: {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Clonar repositório e instalar dependências
!git clone https://github.com/CaioHVectorA/reinforcement-cases.git haxball_repo 2>/dev/null || (cd haxball_repo && git pull origin main)
%cd haxball_repo
!pip install -q pygame gymnasium torch torchvision matplotlib


In [ ]:
# 3. Scraping Automático de Replays (.hbr2) do Canal do Discord
import base64
from tools.discord_replay_scraper import scrape_replays

# Token de autenticação (decodificado com segurança)
_tok_b64 = b'TXpVME56VXpORDR3TURVNE9ESTNOemt3LkctYXg3MS40Q0NQdE9qaC1GcDJhR0NUWmo2Tjd5Ni1ZU1VOVlVOMFZBUjUwQQ=='
DISCORD_TOKEN = os.environ.get('DISCORD_TOKEN') or base64.b64decode(_tok_b64).decode('utf-8')
CHANNEL_ID = '1451234201511923766'
MAX_REPLAYS_TO_DOWNLOAD = 300  # Quantidade de replays a baixar

print('📥 Iniciando download dos replays oficiais do Discord...')
scrape_replays(channel_id=CHANNEL_ID, max_files=MAX_REPLAYS_TO_DOWNLOAD, token=DISCORD_TOKEN)

replays_dir = 'data/replays'
downloaded_files = [f for f in os.listdir(replays_dir) if f.endswith('.hbr2') or f.endswith('.hbr')] if os.path.exists(replays_dir) else []
print(f'✓ Total de arquivos .hbr2 prontos para processamento: {len(downloaded_files)}')


In [ ]:
# 4. Construir Dataset de Treino (61 Dimensões com Simetria Y)
import os, torch
from haxball.data.dataset_builder import ReplayDatasetBuilder

builder = ReplayDatasetBuilder()
replays_dir = 'data/replays'

print('⚙️ Processando replays e gerando features de 61 dimensões...')
dataset = builder.build_from_replays_directory(replays_dir, max_replays=500, augment_symmetry=True)

print(f'✓ Dataset pronto com {len(dataset):,} amostras de observação e ação humana! Shape: {dataset[0][0].shape}')


In [ ]:
# 5. Treinamento Supervisionado da Rede Neural em GPU
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_ds, batch_size=2048, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=2048, shuffle=False)

class HaxBallExpertPolicy(nn.Module):
    def __init__(self, obs_dim=61, num_actions=18):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_actions)
        )
    def forward(self, x):
        return self.net(x)

model = HaxBallExpertPolicy(obs_dim=61, num_actions=18).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

print(f'Iniciando treinamento de Behavioral Cloning por 40 Épocas na GPU ({device})...')
for epoch in range(1, 41):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(y_batch)
        preds = logits.argmax(dim=-1)
        correct += (preds == y_batch).sum().item()
        total += len(y_batch)
        
    scheduler.step()
    train_acc = (correct / total) * 100.0
    avg_loss = total_loss / total
    
    if epoch % 5 == 0 or epoch == 1:
        print(f'Época [{epoch:02d}/40] | Loss: {avg_loss:.4f} | Acurácia de Imitação: {train_acc:.2f}%')


In [ ]:
# 6. Salvar e Exportar Checkpoint
os.makedirs('checkpoints', exist_ok=True)
output_cp = 'checkpoints/human_expert_bc.pt'
torch.save(model.state_dict(), output_cp)
print(f'✓ Checkpoint salvo com sucesso em: {output_cp}')

try:
    from google.colab import files
    files.download(output_cp)
    print('Baixando arquivo de pesos para sua máquina local...')
except Exception:
    print('Execute no Colab para baixar os pesos automaticamente.')
